# Source-grounded Lee Kuan Yew simulation
Select **Runtime > Change runtime type > T4 GPU**, then run cells in order. No API key or Drive mount is required for this public-repository workflow. This notebook is an unexecuted workflow template; executed local results are in the repository. A GPU allocation is subject to your Colab quota.


In [ ]:
import os, pathlib, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training, or use the local CPU instructions.'
print(torch.cuda.get_device_name(0))
workspace = pathlib.Path('/content/99task_lee_kuan_yew_llm')
if not workspace.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DiedrickD/99group-3.git', str(workspace)], check=True)
os.chdir(workspace)
sys.path.insert(0, str(workspace))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)


## Choose a model
Keep 0.5B to reproduce the small experiment. Change to `Qwen/Qwen2.5-1.5B-Instruct` for a larger experiment; rerun preparation, training and evaluation together. Do not compare results across model sizes as if only the adapter changed. The exact model commit is recorded automatically.


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('scripts/validate_data.py')
run('scripts/prepare_model.py', '--model-id', MODEL_ID)
run('-m', 'lky.train', '--epochs', '2')


In [ ]:
run('-m', 'lky.evaluate', '--backend', 'notes', '--output', 'reports/colab-notes.json')
run('-m', 'lky.evaluate', '--backend', 'base', '--output', 'reports/colab-base-rag.json')
run('-m', 'lky.evaluate', '--backend', 'lora', '--output', 'reports/colab-lora-rag.json')
run('-m', 'lky.evaluate', '--backend', 'base', '--no-rag', '--output', 'reports/colab-base-no-rag.json')
run('-m', 'lky.evaluate', '--backend', 'lora', '--no-rag', '--output', 'reports/colab-lora-no-rag.json')


In [ ]:
# Download the trained adapter and measured reports before the runtime expires.
import zipfile
from google.colab import files
archive = '/content/lky-training-artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as output:
    for folder in ['models/lky-lora', 'reports', 'config']:
        for path in pathlib.Path(folder).rglob('*'):
            if path.is_file():
                output.write(path, path.as_posix())
files.download(archive)


## Temporary chat demo
The next cell creates a public Gradio share URL. It works only while this runtime and server remain available. Use the repository chat examples as the permanent submission link. Each turn is independent.


In [ ]:
from app import build_app
demo = build_app('lora')
demo.launch(share=True, debug=False)
